# 01 · Explore the raw data

Before cleaning anything, this notebook checks what is actually in the files from `data/raw/`:

1. Row counts and column types
2. Missing values (nulls)
3. Hidden missing values: `-1` codes that mean "Unknown"

Run `python src/download.py` first to fetch the files.

In [1]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path("..") / "data" / "raw"
pd.set_option("display.width", 120)

## Load the files

The three PriceCatcher months are combined into one table so every check covers all of them.

In [2]:
cpi = pd.read_parquet(RAW_DIR / "cpi_2d_state.parquet")
lookup_item = pd.read_parquet(RAW_DIR / "lookup_item.parquet")
lookup_premise = pd.read_parquet(RAW_DIR / "lookup_premise.parquet")
mcoicop = pd.read_parquet(RAW_DIR / "mcoicop.parquet")

price_files = sorted(RAW_DIR.glob("pricecatcher_*.parquet"))
prices = pd.concat([pd.read_parquet(f) for f in price_files], ignore_index=True)

tables = {
    "prices": prices,
    "cpi": cpi,
    "lookup_item": lookup_item,
    "lookup_premise": lookup_premise,
    "mcoicop": mcoicop,
}
print("PriceCatcher files:", [f.name for f in price_files])

PriceCatcher files: ['pricecatcher_2026-07.parquet', 'pricecatcher_2026-08.parquet', 'pricecatcher_2026-09.parquet']


## 1. Row counts and column types

In [3]:
pd.DataFrame(
    {name: {"rows": len(df), "columns": df.shape[1]} for name, df in tables.items()}
).T

,rows,columns
prices,5491454,4
cpi,45024,4
lookup_item,797,5
lookup_premise,3932,6
mcoicop,343,7


In [4]:
for name, df in tables.items():
    print(f"--- {name}")
    print(df.dtypes.to_string())
    print()

--- prices
date             object
premise_code      int64
item_code         int64
price           float64

--- cpi
state                  str
date        datetime64[ns]
division               str
index              float64

--- lookup_item
item_code        int64
item               str
unit               str
item_group         str
item_category      str

--- lookup_premise
premise_code    int64
premise           str
address           str
premise_type      str
state             str
district          str

--- mcoicop
digits      int64
division      str
group         str
class         str
subclass      str
desc_en       str
desc_bm       str



## 2. Missing values (nulls)

For each table: how many nulls each column has, and what percentage of rows that is.

In [5]:
def null_summary(df):
    """Null count and percentage for every column."""
    nulls = df.isna().sum()
    return pd.DataFrame({"nulls": nulls, "null_pct": (nulls / len(df) * 100).round(2)})


for name, df in tables.items():
    print(f"--- {name}")
    print(null_summary(df).to_string())
    print()

--- prices
              nulls  null_pct
date              0       0.0
premise_code      0       0.0
item_code         0       0.0
price             0       0.0

--- cpi
          nulls  null_pct
state         0       0.0
date          0       0.0
division      0       0.0
index         0       0.0

--- lookup_item
               nulls  null_pct
item_code          0      0.00
item               1      0.13
unit               1      0.13
item_group         1      0.13
item_category      1      0.13

--- lookup_premise
              nulls  null_pct
premise_code      0      0.00
premise           0      0.00
address           0      0.00
premise_type      1      0.03
state             1      0.03
district          1      0.03

--- mcoicop
          nulls  null_pct
digits        0       0.0
division      0       0.0
group         0       0.0
class         0       0.0
subclass      0       0.0
desc_en       0       0.0
desc_bm       0       0.0



## 3. Hidden missing values: `-1` codes

PriceCatcher's lookups use code `-1` for "Unknown". These rows are not null, so step 2 cannot catch them.

In [6]:
print("Lookup rows with code -1:")
print(lookup_item[lookup_item["item_code"] == -1].to_string())
print()
print(lookup_premise[lookup_premise["premise_code"] == -1].to_string())
print()

unknown = pd.DataFrame({
    "rows": [(prices["item_code"] == -1).sum(), (prices["premise_code"] == -1).sum()],
}, index=["item_code == -1", "premise_code == -1"])
unknown["pct_of_prices"] = (unknown["rows"] / len(prices) * 100).round(3)
unknown

Lookup rows with code -1:
   item_code item unit item_group item_category
0         -1  NaN  NaN        NaN           NaN

   premise_code          premise          address premise_type state district
0            -1  Unknown Premise  Unknown Address          NaN   NaN      NaN



,rows,pct_of_prices
item_code == -1,0,0.0
premise_code == -1,0,0.0


## 4. Price outliers

There are no prices of 0 or below, but some prices are clearly typing errors (for example RM 718 for 1 kg of carrots).

**Rule:** within each item, a price is an outlier if it is more than **5x** the item's median price, or less than **1/5** of it.

- The median is used because a single extreme typo barely moves it, unlike the mean.
- The goal is to catch impossible values, not unusual ones. The IQR rule was considered but flags about 3% of rows, including real price differences between premise types, which this project needs to compare.

In [7]:
print("Prices <= 0:", (prices["price"] <= 0).sum())
prices["price"].describe().round(2)

Prices <= 0: 0


count    5491454.00
mean          11.78
std           10.86
min            0.01
25%            5.00
50%            8.99
75%           14.59
max         3900.00
Name: price, dtype: float64

In [8]:
item_median = prices.groupby("item_code")["price"].transform("median")
ratio = prices["price"] / item_median

comparison = pd.DataFrame(
    {"rows_removed": [((ratio > k) | (ratio < 1 / k)).sum() for k in [3, 5, 10]]},
    index=["k = 3", "k = 5", "k = 10"],
)
comparison["pct_of_prices"] = (comparison["rows_removed"] / len(prices) * 100).round(3)
comparison

,rows_removed,pct_of_prices
k = 3,9327,0.170
k = 5,866,0.016
k = 10,282,0.005


The most extreme prices flagged at k = 5, compared with their item's median:

In [9]:
K = 5
is_outlier = (ratio > K) | (ratio < 1 / K)

flagged = prices[is_outlier].assign(item_median=item_median[is_outlier], ratio=ratio[is_outlier])
flagged = flagged.merge(lookup_item[["item_code", "item", "unit"]], on="item_code")

columns = ["item", "unit", "price", "item_median", "ratio"]
print("Highest ratios:")
print(flagged.nlargest(5, "ratio")[columns].round(2).to_string(index=False))
print()
print("Lowest ratios:")
print(flagged.nsmallest(5, "ratio")[columns].round(3).to_string(index=False))

Highest ratios:
                     item  unit  price  item_median  ratio
   KACANG DAL (AUSTRALIA)   1kg 1250.0         4.99 250.50
              LOBAK MERAH   1kg  718.0         3.79 189.45
KUBIS BUNGA (CAULIFLOWER)   1kg 1390.0         7.99 173.97
        LOBAK PUTIH (1KG)   1kg  800.0         4.90 163.27
             UDANG KERING 100 g 1179.0         7.90 149.24

Lowest ratios:
                                                  item unit  price  item_median  ratio
                           MINYAK MASAK TULEN CAP ALIF 5 kg   0.01        30.90  0.000
     IKAN KEMBUNG (ANTARA 8 HINGGA 12 EKOR SEKILOGRAM)  1kg   0.01        16.00  0.001
IKAN TILAPIA MERAH (ANTARA 2 HINGGA 5 EKOR SEKILOGRAM)  1kg   0.01        16.00  0.001
                                   CILI MERAH - MINYAK  1kg   0.02        12.99  0.002
                                           LOBAK MERAH  1kg   0.01         3.79  0.003


## 5. Items with few prices

A per-item median is only reliable when an item has enough prices. Items with fewer than 30 prices across the three months are listed here, so they can be avoided when choosing basket items.

In [10]:
item_counts = prices.groupby("item_code").size().rename("price_rows")
few = (
    item_counts[item_counts < 30]
    .reset_index()
    .merge(lookup_item[["item_code", "item", "unit"]], on="item_code")
    .sort_values("price_rows")
)
print(f"{len(few)} of {len(item_counts)} items have fewer than 30 prices")
print(few.to_string(index=False))

9 of 343 items have fewer than 30 prices
 item_code  price_rows                                                    item    unit
      1399           1                            BAWANG PERAI (LEEK) TEMPATAN     1kg
      1419           7                                KACANG DAL BIASA (INDIA)     1kg
       118          11 TELUR AYAM GRED A (BERAT 65.0 GM HINGGA 69.9 GM SEBIJI) 10 biji
      2037          11                              BIHUN KERING (CAP LONGKOU)   400 g
       120          13 TELUR AYAM GRED C (BERAT 55.0 GM HINGGA 59.9 GM SEBIJI) 10 biji
       119          18        TELUR AYAM GRED B (BERAT 60.0 GM HINGGA 64.9 GM) 10 biji
      1482          25                                KUBIS PANJANG (TEMPATAN)     1kg
      1420          26                               KACANG DAL MALAVI (INDIA)     1kg
      2004          26                           BERAS SUPER IMPORT CAP FAMILY   10 kg


## 6. Units

Prices are not converted to a common unit (such as per kg). Every research question compares the **same item** over time, across premise types or across states, and each item has one fixed unit, so its prices are already comparable. Many units also cannot be converted to weight, such as `10 biji`, `1 liter` or `M64, L54, XL42`.

`unit` is kept as a label that describes what one price buys.

In [11]:
print("Each item_code has exactly one unit:", lookup_item["item_code"].is_unique)

used_items = lookup_item[lookup_item["item_code"].isin(prices["item_code"].unique())]
print(f"{len(used_items)} items appear in the prices, using {used_items['unit'].nunique()} different unit labels")
print()
print(used_items["unit"].value_counts().head(10).to_string())

Each item_code has exactly one unit: True
329 items appear in the prices, using 75 different unit labels

unit
1kg        147
10 kg       17
250 g       10
2 kg        10
400 g        8
5 kg         7
500 g        7
850g         6
1 liter      5
340 g        4


## 7. Stray spaces in text

`premise_type` has two versions of `Pasar Basah` (wet market), so wet markets would be split into two groups. The check below finds every text value with leading or trailing spaces.

In [12]:
print("Pasar Basah values before trimming:")
print(lookup_premise.loc[lookup_premise["premise_type"].str.contains("Basah", na=False), "premise_type"].map(repr).value_counts().to_string())
print()
print("After trimming:")
print(lookup_premise["premise_type"].str.strip().loc[lambda s: s.str.contains("Basah", na=False)].value_counts().to_string())

Pasar Basah values before trimming:
premise_type
'Pasar Basah '    338
'Pasar Basah'      22

After trimming:
premise_type
Pasar Basah    360


In [13]:
rows = []
for name, df in tables.items():
    for col in df.select_dtypes(include=["object", "string"]).columns:
        values = df[col].dropna().astype(str)
        rows.append({
            "table": name,
            "column": col,
            "values_with_spaces": (values != values.str.strip()).sum(),
            "distinct_before": values.nunique(),
            "distinct_after": values.str.strip().nunique(),
        })

spaces = pd.DataFrame(rows)
print(spaces[spaces["values_with_spaces"] > 0].to_string(index=False))

         table       column  values_with_spaces  distinct_before  distinct_after
lookup_premise premise_type                 338               13              12
       mcoicop      desc_en                   1              293             292


## Summary

**What the data looks like**
- **5,491,454 PriceCatcher rows** (July to September 2026) and **45,024 CPI rows**, with **no nulls** in either.
- The only lookup nulls are the `-1` "Unknown" rows, and **no price row uses a `-1` code**.
- `mcoicop` uses `-` instead of null for code levels that don't apply; only the 2-digit divisions are needed.
- PriceCatcher `date` is stored as text.
- **No prices of 0 or below**, but there are clear typing errors, such as RM 718 for 1 kg of carrots (median RM 3.79) and RM 0.01 for 5 kg of cooking oil (median RM 30.90).
- Each item has **exactly one unit**; 329 items use 75 different unit labels, many of which cannot be converted to weight.
- `premise_type` has `Pasar Basah` twice because of a trailing space; one `mcoicop` description also has a stray space.
- **9 items have fewer than 30 prices**, including graded eggs A/B/C.
- October 2026 was dropped because its file only covers 10–31 October; August and September each miss one or two days.

**Cleaning rules** (applied in the SQL clean layer in Phase 4; raw files stay unchanged)

1. **Months:** use July to September 2026, complete months that match the last three months of CPI.
2. **Unknown codes:** drop the `-1` rows from `lookup_item` and `lookup_premise`.
3. **Dates:** convert PriceCatcher `date` from text to a real date.
4. **Outliers:** remove prices more than 5x or less than 1/5 of their item's median. This removes 866 rows (0.016%). k = 3 would remove 9,327 rows, starting to include plausible prices; k = 10 would keep obvious errors.
5. **Text:** trim leading and trailing spaces in all text columns, which merges the two `Pasar Basah` values into one (360 premises).
6. **Units:** keep `unit` as a label and do not convert prices, because every analysis compares the same item.
7. **CPI categories:** use only the 2-digit `mcoicop` divisions `01`–`13`, read as text so the leading zero is kept.

**Notes for later phases**
- For the Q4 basket, choose items with plenty of prices and clear units (such as `1kg` or `10 biji`), avoiding the 9 low-count items.